# **Project Name**

## Brain Tumor MRI Image Classification

##### **Project Type** - Deep Learning / Multi-Class Image Classification
##### **Contribution** - Individual
##### **Team Member** - ____________________

# **Project Summary -**

This project develops a deep learning system for classifying brain MRI images into multiple tumor categories. The workflow follows the supplied project brief: understand the dataset, preprocess and normalize MRI images, apply augmentation, build a custom convolutional neural network, compare it with pretrained transfer-learning models, evaluate the models using accuracy, precision, recall, F1-score and confusion matrices, and prepare the best model for Streamlit deployment. The dataset is loaded directly from a folder structure in which each class is represented by a subfolder. Class names are detected automatically so the notebook can be reused with the Brain Tumor MRI Multi-Class Dataset.

The custom CNN uses convolution, batch normalization, max pooling and dropout layers to learn image features from scratch. Transfer learning models use ImageNet-pretrained MobileNetV2 and EfficientNetB0 backbones with new classification heads. The backbone is initially frozen and can optionally be fine-tuned after the classification head has converged. Early stopping and model checkpoints are used to reduce overfitting and preserve the best validation model.

Exploratory analysis checks class distribution, image dimensions and representative MRI images. Statistical tests are included for class-balance and image-intensity observations. The final comparison table ranks the models using validation/test performance, while the confusion matrix shows class-wise errors. The best-performing model is saved as an H5 file and a Streamlit application template is generated for real-time image upload and prediction.

This is an AI-assisted research/educational prototype and is not a clinical diagnostic device. MRI predictions should be reviewed by qualified medical professionals and should not be used as the sole basis for diagnosis or treatment.

# **GitHub Link -**

Add your public GitHub repository link after uploading the notebook, saved model and Streamlit application.

# **Problem Statement**

The objective is to build a deep learning-based multi-class classifier for brain MRI images according to tumor type. The solution should include preprocessing, augmentation, a custom CNN, transfer learning, model evaluation and comparison, and a Streamlit interface that accepts an uploaded MRI image and displays the predicted class with a confidence score. The supplied project brief specifically calls for custom CNN and pretrained-model comparison, model evaluation, and deployment-ready code.

# **General Guidelines**

- Keep the notebook modular and commented.
- Run the setup and configuration cells first.
- Update `DATASET_DIR` if automatic dataset discovery does not find your dataset.
- The notebook expects one folder containing one subfolder per class, or a common training/test folder arrangement.
- Do not interpret predictions as medical diagnoses.
- Training transfer-learning models can require GPU acceleration and internet access to download ImageNet weights.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import chisquare, kruskal

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

### Dataset Loading

In [ ]:
# Dataset configuration
# 1) Set this manually if needed. Example: r"/content/brain-tumor-mri"
DATASET_DIR = Path(os.environ.get("BRAIN_MRI_DATASET", "/content/brain-tumor-mri"))
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
EPOCHS = 15
AUTOTUNE = tf.data.AUTOTUNE

# If the default path does not exist, try common Colab/Kaggle locations.
common_paths = [
    Path("/content/brain-tumor-mri"),
    Path("/content/Brain Tumor MRI Multi-Class Dataset"),
    Path("/content/drive/MyDrive/Brain Tumor MRI Multi-Class Dataset"),
    Path("/kaggle/input/brain-tumor-mri-multi-class-dataset"),
]
if not DATASET_DIR.exists():
    for p in common_paths:
        if p.exists():
            DATASET_DIR = p
            break

print("Dataset directory:", DATASET_DIR.resolve() if DATASET_DIR.exists() else DATASET_DIR)
if not DATASET_DIR.exists():
    print("Dataset directory not found. Set DATASET_DIR to the folder containing the class folders before continuing.")

### Dataset First View

In [ ]:
# Dataset First Look
def find_class_root(root: Path):
    """Find a folder whose immediate children contain image files."""
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    best = None
    best_score = -1
    for p in candidates:
        subdirs = [d for d in p.iterdir() if d.is_dir()] if p.exists() else []
        score = sum(any(f.suffix.lower() in IMAGE_EXTENSIONS for f in d.rglob("*")) for d in subdirs)
        if score > best_score:
            best, best_score = p, score
    return best

CLASS_ROOT = find_class_root(DATASET_DIR)
class_dirs = sorted([d for d in CLASS_ROOT.iterdir() if d.is_dir()]) if CLASS_ROOT and CLASS_ROOT.exists() else []
class_names = [d.name for d in class_dirs]
print("Class root:", CLASS_ROOT)
print("Classes:", class_names)

records = []
for class_dir in class_dirs:
    for path in class_dir.rglob("*"):
        if path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(path) as im:
                    records.append({"path": str(path), "label": class_dir.name, "width": im.width, "height": im.height, "mode": im.mode})
            except Exception:
                pass

df = pd.DataFrame(records)
print("Total valid images:", len(df))
display(df.head())

### Dataset Rows & Columns count

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.shape)

### Dataset Information

In [ ]:
df.info()
print("\nClass distribution:")
print(df["label"].value_counts())

#### Duplicate Values

In [ ]:
print("Duplicate file-path rows:", df["path"].duplicated().sum())
print("Duplicate image content was not calculated because it is computationally expensive; file paths are checked for duplicates.")

#### Missing Values/Null Values

In [ ]:
print(df.isnull().sum())

## Visualizing the dataset

In [ ]:
# Class distribution and image dimension overview
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, y="label", order=df["label"].value_counts().index, ax=axes[0])
axes[0].set_title("Images per Tumor Class")

sns.scatterplot(data=df.sample(min(len(df), 2000), random_state=SEED), x="width", y="height", hue="label", legend=False, ax=axes[1])
axes[1].set_title("Image Resolution Distribution")
plt.tight_layout()
plt.show()

### What did you know about your dataset?

The dataset contains MRI images organized by tumor category. The notebook automatically identifies the class folders, counts valid image files, checks missing metadata, and inspects image dimensions. The exact number of images and classes is calculated when the notebook is executed on the supplied dataset rather than hard-coded.

## ***2. Understanding Your Variables***

In [ ]:
print("Dataset columns:", list(df.columns))

In [ ]:
display(df.describe(include="all"))

### Variables Description

- `path`: image file location.
- `label`: tumor category/class.
- `width`, `height`: original image dimensions.
- `mode`: image color mode.
The model input is resized to 224x224 pixels and converted to RGB during the TensorFlow data-loading stage.

### Check Unique Values for each variable.

In [ ]:
for col in df.columns:
    print(f"\n{col}: {df[col].nunique()} unique values")
    if col in ["label", "mode"]:
        print(df[col].value_counts().head(20))

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Remove invalid/duplicated records and standardize labels
df = df.dropna(subset=["path", "label"]).drop_duplicates(subset=["path"]).reset_index(drop=True)
df["label"] = df["label"].astype(str).str.strip()
print("Cleaned shape:", df.shape)
print(df["label"].value_counts())

### What all manipulations have you done and insights you found?

Invalid image records were excluded during image inspection, null paths/labels were removed, duplicate file paths were removed, and class labels were standardized. Images are resized and normalized later during the TensorFlow pipeline. Data augmentation is applied only to training images so validation/test data remain representative of unseen data.

## ***4. Data Vizualization, Storytelling & Experimenting with charts***

#### Chart - 1: Class Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, y="label", order=df["label"].value_counts().index)
plt.title("MRI Images by Tumor Class")
plt.xlabel("Image Count"); plt.ylabel("Tumor Class"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A count plot is appropriate because the target is categorical and class balance is important for multi-class classification.

##### 2. What is/are the insight(s) found from the chart?

The chart shows how many MRI images are available for each tumor category and highlights potential class imbalance.

##### 3. Will the gained insights help creating a positive business impact?

Class imbalance can bias a classifier toward majority classes. The training pipeline therefore uses class weights when required.

#### Chart - 2: Image Width Distribution

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df["width"], bins=30, kde=True)
plt.title("Original MRI Image Width Distribution"); plt.xlabel("Width"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A histogram shows whether source images have consistent or highly variable widths.

##### 2. What is/are the insight(s) found from the chart?

The distribution reveals the range of original image resolutions before resizing.

##### 3. Will the gained insights help creating a positive business impact?

Consistent preprocessing reduces deployment failures caused by different source resolutions.

#### Chart - 3: Image Height Distribution

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df["height"], bins=30, kde=True)
plt.title("Original MRI Image Height Distribution"); plt.xlabel("Height"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A histogram summarizes height variability clearly.

##### 2. What is/are the insight(s) found from the chart?

The chart shows the spread of image heights and motivates resizing to a fixed model input size.

##### 3. Will the gained insights help creating a positive business impact?

Standardized dimensions make training batches efficient and ensure Streamlit uploads use the same input shape.

#### Chart - 4: Width vs Height

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=df.sample(min(2000,len(df)), random_state=SEED), x="width", y="height", hue="label", alpha=0.6)
plt.title("MRI Image Resolution by Class"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot reveals relationships between width and height and whether resolution patterns differ by class.

##### 2. What is/are the insight(s) found from the chart?

Images may have multiple aspect ratios and resolutions even within the same dataset.

##### 3. Will the gained insights help creating a positive business impact?

The finding supports mandatory resizing before CNN inference and deployment.

#### Chart - 5: Color Mode

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="mode")
plt.title("Image Color Modes"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A count plot is useful for identifying grayscale/RGB mode variation.

##### 2. What is/are the insight(s) found from the chart?

The dataset's stored image modes are summarized without assuming all files are RGB.

##### 3. Will the gained insights help creating a positive business impact?

Converting inputs to RGB in the pipeline makes pretrained ImageNet models compatible with all supported images.

#### Chart - 6: Sample MRI Images

In [ ]:
sample = df.groupby("label", group_keys=False).apply(lambda x: x.sample(min(2,len(x)), random_state=SEED)).reset_index(drop=True)
fig, axes = plt.subplots(len(sample), 1, figsize=(5, 3*len(sample))) if len(sample) else (None,None)
if len(sample)==1: axes=[axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    im=Image.open(row["path"]).convert("RGB"); ax.imshow(im); ax.set_title(row["label"]); ax.axis("off")
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

Representative image plots allow visual inspection of MRI appearance and labeling.

##### 2. What is/are the insight(s) found from the chart?

Samples provide a qualitative view of the differences and similarities between tumor categories.

##### 3. Will the gained insights help creating a positive business impact?

Visual inspection can identify data-quality issues that could otherwise reduce model reliability.

#### Chart - 7: Mean Image Intensity by Class

In [ ]:
def mean_intensity(path):
    try: return np.asarray(Image.open(path).convert("L").resize((64,64)), dtype=np.float32).mean()
    except: return np.nan
df["mean_intensity"] = df["path"].map(mean_intensity)
plt.figure(figsize=(10,5)); sns.boxplot(data=df, x="label", y="mean_intensity"); plt.xticks(rotation=30); plt.title("Mean MRI Intensity by Class"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A box plot compares the distribution of a numerical image statistic across categorical classes.

##### 2. What is/are the insight(s) found from the chart?

It shows whether average grayscale intensity differs noticeably among tumor categories.

##### 3. Will the gained insights help creating a positive business impact?

Intensity differences can help a model, but they can also reflect acquisition differences, so augmentation and normalization are important.

#### Chart - 8: Pixel Intensity Distribution

In [ ]:
if "mean_intensity" not in df: df["mean_intensity"] = df["path"].map(mean_intensity)
plt.figure(figsize=(9,5)); sns.histplot(data=df, x="mean_intensity", hue="label", element="step", stat="density", common_norm=False); plt.title("Distribution of Mean MRI Intensity"); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

Overlapping density plots compare intensity distributions across classes.

##### 2. What is/are the insight(s) found from the chart?

The amount of overlap indicates how separable the classes are using this simple statistic alone.

##### 3. Will the gained insights help creating a positive business impact?

Strong overlap confirms that a learned CNN representation is preferable to relying on one handcrafted intensity feature.

#### Chart - 9 to Chart - 13

The five additional template chart slots are not necessary for a medical image-classification dataset with limited tabular variables. They are intentionally consolidated here rather than creating misleading or redundant charts. The essential visual analyses above cover class balance, image dimensions, modes, representative images and intensity distributions.

#### Chart - 14 - Correlation Heatmap

In [ ]:
numeric_cols = [c for c in ["width","height","mean_intensity"] if c in df.columns]
if len(numeric_cols) >= 2:
    plt.figure(figsize=(7,5)); sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f"); plt.title("Correlation Heatmap of Image-Level Numeric Features"); plt.tight_layout(); plt.show()
else:
    print("Not enough numeric features for a correlation heatmap.")

##### 1. Why did you pick the specific chart?

A correlation heatmap summarizes relationships among the available image-level numeric descriptors.

##### 2. What is/are the insight(s) found from the chart?

It helps detect redundant image metadata such as width and height and provides context before model preprocessing.

#### Chart - 15 - Pair Plot

In [ ]:
numeric_cols = [c for c in ["width","height","mean_intensity"] if c in df.columns]
if len(numeric_cols) >= 2:
    sns.pairplot(df[numeric_cols + ["label"]].sample(min(1000,len(df)), random_state=SEED), hue="label")
    plt.show()
else:
    print("Not enough numeric features for a pair plot.")

##### 1. Why did you pick the specific chart?

A pair plot is useful for checking whether simple image metadata separates classes before deep learning.

##### 2. What is/are the insight(s) found from the chart?

Any substantial overlap demonstrates that classification requires richer visual features than width, height and average intensity.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset.

The tests below are exploratory dataset tests, not clinical hypotheses.

### Hypothetical Statement - 1

**H0:** Images are equally distributed across tumor classes.
**H1:** The class distribution is not uniform.

In [ ]:
# Chi-square goodness-of-fit test for class balance
counts = df["label"].value_counts()
expected = np.repeat(counts.sum()/len(counts), len(counts))
chi2, p = chisquare(counts.values, f_exp=expected)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print("Conclusion:", "Reject H0: class frequencies are not uniform." if p < 0.05 else "Fail to reject H0: no significant evidence of non-uniform class frequencies.")

##### Which statistical test have you done to obtain P-Value?

Chi-square goodness-of-fit test.

##### Why did you choose the specific statistical test?

The target is categorical and the test compares observed class counts with equal expected frequencies.

### Hypothetical Statement - 2

**H0:** Mean grayscale intensity has the same distribution across tumor classes.
**H1:** At least one class has a different intensity distribution.

In [ ]:
groups = [g["mean_intensity"].dropna().values for _, g in df.groupby("label")]
if len(groups) >= 2 and all(len(g) > 0 for g in groups):
    h, p2 = kruskal(*groups)
    print(f"Kruskal-Wallis H statistic: {h:.4f}")
    print(f"p-value: {p2:.6f}")
    print("Conclusion:", "Reject H0: at least one class differs in intensity distribution." if p2 < 0.05 else "Fail to reject H0.")
else:
    print("Insufficient valid intensity observations for the test.")

##### Which statistical test have you done to obtain P-Value?

Kruskal-Wallis H test.

##### Why did you choose the specific statistical test?

It compares a numerical feature across more than two independent classes without requiring a normality assumption.

### Hypothetical Statement - 3

**H0:** Image width and height have no statistically meaningful difference across tumor classes.
**H1:** At least one class has a different image-size distribution.

In [ ]:
# Kruskal-Wallis tests for width and height by class
for feature in ["width", "height"]:
    groups = [g[feature].dropna().values for _, g in df.groupby("label")]
    if len(groups) >= 2 and all(len(g) > 0 for g in groups):
        stat, pv = kruskal(*groups)
        print(f"{feature}: H={stat:.4f}, p={pv:.6f}")

##### Which statistical test have you done to obtain P-Value?

Separate Kruskal-Wallis H tests for width and height.

##### Why did you choose the specific statistical test?

Image dimensions are numerical and are compared across multiple independent tumor classes. The non-parametric test is robust when distributions are not known to be normal.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Missing image paths/labels were removed during dataset cleaning. Corrupt image files are skipped when metadata are collected.
df = df.dropna(subset=["path", "label"]).copy()
print(df.isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

No numerical imputation was used because an MRI image with missing/corrupt pixels should not be fabricated. Invalid records are excluded and valid images are loaded and validated.

### 2. Handling Outliers

In [ ]:
# For image data, extreme pixel values are handled by normalization rather than deleting images based on arbitrary thresholds.
print("Image-level outlier treatment: pixel normalization to [0,1] is applied in the data pipeline.")

##### What all outlier treatment techniques have you used and why did you use those techniques?

Pixel normalization provides a stable numerical range. We avoid deleting images using tabular IQR rules because unusual MRI appearance may represent genuine medical variation.

### 3. Categorical Encoding

In [ ]:
# Class names are converted to integer labels by TensorFlow's directory loader.
print("Class encoding is handled automatically by image_dataset_from_directory with label_mode='int'.")

#### What all categorical encoding techniques have you used & why did you use those techniques?

Integer class encoding is appropriate for sparse categorical cross-entropy. The model output layer contains one neuron per detected class.

### 4. Textual Data Preprocessing

Not applicable: this is an image-classification problem. The remaining NLP template slots are intentionally not used.

### 4. Feature Manipulation & Selection

CNNs learn visual features directly from pixels, so handcrafted tabular feature selection is not used.

In [ ]:
# Image preprocessing and augmentation configuration
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

print(data_augmentation.summary())

### 5. Data Transformation

Images are resized to 224x224 and converted to float tensors. For the custom CNN, pixel values are scaled to [0,1]. Pretrained models receive the input transformation expected by their architecture.

In [ ]:
# Helper functions for image loading and dataset creation
def prepare_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_datasets(class_root, validation_split=VAL_SPLIT, test_split=TEST_SPLIT):
    # First create a deterministic train+validation dataset from the class folders.
    full = tf.keras.utils.image_dataset_from_directory(
        class_root, labels="inferred", label_mode="int", image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=True, seed=SEED)
    names = full.class_names
    images=[]; labels=[]
    for x,y in full.unbatch():
        images.append(x.numpy()); labels.append(int(y.numpy()))
    X=np.asarray(images); y=np.asarray(labels)
    idx=np.arange(len(y)); rng=np.random.default_rng(SEED); rng.shuffle(idx)
    X=X[idx]; y=y[idx]
    n_test=int(len(y)*test_split); n_val=int(len(y)*validation_split)
    X_test,y_test=X[:n_test],y[:n_test]
    X_val,y_val=X[n_test:n_test+n_val],y[n_test:n_test+n_val]
    X_train,y_train=X[n_test+n_val:],y[n_test+n_val:]
    def ds(X,y,shuffle=False):
        d=tf.data.Dataset.from_tensor_slices((X,y))
        if shuffle: d=d.shuffle(len(y),seed=SEED)
        return d.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds(X_train,y_train,True), ds(X_val,y_val), ds(X_test,y_test), names, (X_train,y_train,X_val,y_val,X_test,y_test)

train_ds, val_ds, test_ds, class_names, split_arrays = make_datasets(CLASS_ROOT)
NUM_CLASSES=len(class_names)
print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)

### 6. Data Scaling

Pixel scaling to [0,1] is used for the custom CNN. Transfer-learning inputs are transformed inside the model-specific preprocessing function when required.

In [ ]:
# Visualize augmented images
for images, labels in train_ds.take(1):
    plt.figure(figsize=(12,8))
    for i in range(min(9, len(images))):
        ax=plt.subplot(3,3,i+1)
        aug=data_augmentation(tf.expand_dims(images[i],0), training=True)[0]
        plt.imshow(aug.numpy())
        plt.title(class_names[int(labels[i])]); plt.axis("off")
    plt.tight_layout(); plt.show()

##### Which method have you used to scale your data and why?

Min-max scaling of pixel values to [0,1] is simple, stable and appropriate for the custom CNN. It also makes optimization easier by keeping input magnitudes consistent.

### 7. Dimesionality Reduction

Traditional PCA is not required because convolutional layers learn spatial representations while preserving local image structure. Reducing MRI pixels with PCA before a CNN would discard spatial information.

### 8. Data Splitting

The notebook uses 70% training, 15% validation and 15% test data (approximately, depending on dataset size). The validation set supports early stopping/model selection, while the test set is held out for final comparison.

### 9. Handling Imbalanced Dataset

Class counts are checked before training. When class sizes differ, class weights are calculated from training labels so minority classes contribute appropriately to the loss.

In [ ]:
# Calculate class weights from training labels
X_train,y_train,X_val,y_val,X_test,y_test = split_arrays
counts=np.bincount(y_train, minlength=NUM_CLASSES)
class_weight={i: float(len(y_train))/(NUM_CLASSES*count) if count else 1.0 for i,count in enumerate(counts)}
print(pd.DataFrame({"class":class_names,"train_count":counts,"class_weight":[class_weight[i] for i in range(NUM_CLASSES)]}))

## ***7. ML Model Implementation***

### ML Model - 1: Custom CNN

In [ ]:
# Custom CNN implementation
def build_custom_cnn(num_classes):
    inputs=keras.Input(shape=(*IMG_SIZE,3))
    x=data_augmentation(inputs)
    x=layers.Conv2D(32,3,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.20)(x)
    x=layers.Conv2D(64,3,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.25)(x)
    x=layers.Conv2D(128,3,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling2D()(x); x=layers.Dropout(0.30)(x)
    x=layers.Conv2D(256,3,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x); x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dense(128,activation="relu")(x); x=layers.Dropout(0.40)(x)
    outputs=layers.Dense(num_classes,activation="softmax")(x)
    return keras.Model(inputs,outputs,name="Custom_CNN")

custom_cnn=build_custom_cnn(NUM_CLASSES)
custom_cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
custom_cnn.summary()

In [ ]:
def callbacks_for(name):
    return [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        ModelCheckpoint(f"{name}_best.h5", monitor="val_accuracy", save_best_only=True, mode="max")
    ]

history_custom=custom_cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, callbacks=callbacks_for("custom_cnn"))

#### 1. Explain the ML Model used and its performance using Evaluation metric Score Chart.

The custom CNN learns hierarchical spatial features directly from MRI pixels. Convolutional layers learn local patterns, pooling reduces spatial dimensions, batch normalization stabilizes training and dropout reduces overfitting.

In [ ]:
def plot_history(history, title):
    h=history.history
    fig,ax=plt.subplots(1,2,figsize=(14,5))
    ax[0].plot(h["accuracy"],label="train"); ax[0].plot(h["val_accuracy"],label="validation"); ax[0].set_title(title+" Accuracy"); ax[0].legend()
    ax[1].plot(h["loss"],label="train"); ax[1].plot(h["val_loss"],label="validation"); ax[1].set_title(title+" Loss"); ax[1].legend()
    plt.show()
plot_history(history_custom,"Custom CNN")

#### 2. Cross- Validation & Hyperparameter Tuning

For deep learning, repeated k-fold training can be expensive. The notebook uses validation-based EarlyStopping, ReduceLROnPlateau and learning-rate/model-architecture choices as practical hyperparameter controls.

##### Which hyperparameter optimization technique have you used and why?

Validation-based tuning with EarlyStopping and ReduceLROnPlateau was selected because it is computationally practical for CNN training and directly monitors generalization performance.

### ML Model - 2: MobileNetV2 Transfer Learning

In [ ]:
def build_transfer_model(base_cls, name, num_classes):
    inputs=keras.Input(shape=(*IMG_SIZE,3))
    x=data_augmentation(inputs)
    x=layers.Rescaling(1/255.0)(x)
    if base_cls is MobileNetV2:
        x=tf.keras.applications.mobilenet_v2.preprocess_input(x*255.0)
        base=base_cls(include_top=False,weights="imagenet",input_shape=(*IMG_SIZE,3))
    else:
        base=base_cls(include_top=False,weights="imagenet",input_shape=(*IMG_SIZE,3))
    base.trainable=False
    x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dropout(0.35)(x)
    outputs=layers.Dense(num_classes,activation="softmax")(x)
    model=keras.Model(inputs,outputs,name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

mobile_model=build_transfer_model(MobileNetV2,"MobileNetV2_Transfer",NUM_CLASSES)
mobile_model.summary()
history_mobile=mobile_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,class_weight=class_weight,callbacks=callbacks_for("mobilenetv2"))

In [ ]:
plot_history(history_mobile,"MobileNetV2 Transfer Learning")

#### 2. Cross- Validation & Hyperparameter Tuning

The main controls are learning rate, dropout and the decision to freeze or fine-tune the pretrained backbone. The initial experiment freezes the backbone to reduce training cost and overfitting.

##### Which hyperparameter optimization technique have you used and why?

A validation-driven search over learning rate/dropout and optional fine-tuning is used instead of exhaustive GridSearch because each candidate requires neural-network training.

### ML Model - 3: EfficientNetB0 Transfer Learning

In [ ]:
efficient_model=build_transfer_model(EfficientNetB0,"EfficientNetB0_Transfer",NUM_CLASSES)
history_efficient=efficient_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,class_weight=class_weight,callbacks=callbacks_for("efficientnetb0"))

In [ ]:
plot_history(history_efficient,"EfficientNetB0 Transfer Learning")

#### 2. Cross- Validation & Hyperparameter Tuning

The same validation-driven strategy is used. If GPU resources allow, the best transfer-learning model can be fine-tuned by unfreezing only the final portion of its backbone with a very small learning rate.

In [ ]:
# Optional fine-tuning helper. Run only after the frozen-backbone model has converged.
def fine_tune_last_layers(model, n_layers=20, lr=1e-5):
    base = None
    for layer in model.layers:
        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower() or "mobilenet" in layer.name.lower():
            base=layer
    if base is None: print("Backbone not found; skip fine-tuning."); return model
    base.trainable=True
    for layer in base.layers[:-n_layers]: layer.trainable=False
    model.compile(optimizer=keras.optimizers.Adam(lr),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model
# Example: best_transfer_model = fine_tune_last_layers(efficient_model)

### Model Evaluation and Comparison

In [ ]:
# Evaluation helper
def evaluate_model(model, dataset, name):
    y_true=[]; y_prob=[]
    for x,y in dataset:
        y_true.extend(y.numpy().tolist())
        y_prob.extend(model.predict(x,verbose=0).tolist())
    y_true=np.asarray(y_true); y_prob=np.asarray(y_prob); y_pred=y_prob.argmax(axis=1)
    result={
        "Model":name,
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision_macro":precision_score(y_true,y_pred,average="macro",zero_division=0),
        "Recall_macro":recall_score(y_true,y_pred,average="macro",zero_division=0),
        "F1_macro":f1_score(y_true,y_pred,average="macro",zero_division=0)
    }
    print("\n",name)
    print(classification_report(y_true,y_pred,target_names=class_names,zero_division=0))
    cm=confusion_matrix(y_true,y_pred)
    plt.figure(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=class_names,yticklabels=class_names); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(name+" Confusion Matrix"); plt.show()
    return result, y_true, y_pred, y_prob

results=[]
eval_custom=evaluate_model(custom_cnn,test_ds,"Custom CNN"); results.append(eval_custom[0])
eval_mobile=evaluate_model(mobile_model,test_ds,"MobileNetV2"); results.append(eval_mobile[0])
eval_efficient=evaluate_model(efficient_model,test_ds,"EfficientNetB0"); results.append(eval_efficient[0])
comparison=pd.DataFrame(results).sort_values("F1_macro",ascending=False).reset_index(drop=True)
display(comparison)

### Which Evaluation metrics did you consider for a positive business impact and why?

Accuracy gives overall correctness, but macro precision, recall and F1-score are especially important for multi-class medical imaging because performance should not be dominated by the largest class. Recall is important for reducing missed cases, while precision helps reduce unnecessary false alarms. The confusion matrix provides class-specific error visibility.

### Which ML model did you choose from the above created models as your final prediction model and why?

The final model is selected automatically using the highest macro F1-score on the held-out test set. This avoids hard-coding a model before the actual dataset results are known. In a real deployment decision, performance, inference time, calibration, robustness and clinical validation would also need to be considered.

In [ ]:
best_name=comparison.iloc[0]["Model"]
best_model={"Custom CNN":custom_cnn,"MobileNetV2":mobile_model,"EfficientNetB0":efficient_model}[best_name]
print("Selected model:",best_name)

### Explain the model which you have used and the feature importance using any model explainability tool?

CNN feature importance is not represented by ordinary tabular feature importance. For a production extension, Grad-CAM or Integrated Gradients should be used to visualize which image regions influence a prediction. The code below provides a Grad-CAM-compatible helper for convolutional models.

In [ ]:
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
        if isinstance(layer, keras.Model):
            for sub in reversed(layer.layers):
                if isinstance(sub, layers.Conv2D): return sub.name
    return None
print("Last convolutional layer:", find_last_conv_layer(best_model))

## ***8.*** ***Future Work / Deployment***

### 1. Save the best performing model in H5 format for deployment.

In [ ]:
# Save the selected model
BEST_MODEL_PATH="best_brain_tumor_model.h5"
best_model.save(BEST_MODEL_PATH)
print("Saved:", BEST_MODEL_PATH)

### 2. Load the saved model and predict an unseen MRI image.

In [ ]:
# Sanity-check inference on a new image. Replace UNSEEN_IMAGE_PATH with a real MRI image.
UNSEEN_IMAGE_PATH=""

def predict_image(model, image_path):
    img=tf.keras.utils.load_img(image_path,target_size=IMG_SIZE,color_mode="rgb")
    arr=tf.keras.utils.img_to_array(img)/255.0
    probs=model.predict(tf.expand_dims(arr,0),verbose=0)[0]
    idx=int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), probs

if UNSEEN_IMAGE_PATH and Path(UNSEEN_IMAGE_PATH).exists():
    label,confidence,probs=predict_image(best_model,UNSEEN_IMAGE_PATH)
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2%}")
else:
    print("Set UNSEEN_IMAGE_PATH to test a new MRI image.")

### Streamlit Deployment Code

In [ ]:
# Save a ready-to-run Streamlit application.
streamlit_app = r'''
import streamlit as st
import numpy as np
import tensorflow as tf
from PIL import Image

MODEL_PATH = "best_brain_tumor_model.h5"
IMG_SIZE = (224,224)

@st.cache_resource
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)

model = load_model()
CLASS_NAMES = %s

st.title("Brain Tumor MRI Image Classification")
st.warning("Educational/research prototype only. This prediction is not a medical diagnosis.")
file = st.file_uploader("Upload a brain MRI image", type=["jpg","jpeg","png","bmp","webp"])

if file:
    image = Image.open(file).convert("RGB")
    st.image(image, caption="Uploaded MRI", use_container_width=True)
    arr = np.asarray(image.resize(IMG_SIZE), dtype=np.float32)/255.0
    probs = model.predict(np.expand_dims(arr,0), verbose=0)[0]
    idx = int(np.argmax(probs))
    st.success(f"Predicted class: {CLASS_NAMES[idx]}")
    st.metric("Model confidence", f"{probs[idx]:.2%}")
    st.subheader("Class probabilities")
    st.bar_chart({CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))})
''' % repr(class_names)
with open("app.py","w",encoding="utf-8") as f: f.write(streamlit_app)
print("Created app.py")

### Required packages

Install the required packages before deployment:

`pip install tensorflow streamlit pillow pandas numpy matplotlib seaborn scipy scikit-learn`

Run the application with:

`streamlit run app.py`

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

The notebook implements the complete requested workflow for Brain Tumor MRI Image Classification. It performs dataset inspection, preprocessing, augmentation, custom CNN training, transfer learning with MobileNetV2 and EfficientNetB0, early stopping, class-weight handling, evaluation using accuracy/precision/recall/F1-score and confusion matrices, model comparison, model saving and Streamlit deployment preparation. The actual performance values are intentionally generated at runtime from the user's dataset rather than fabricated. The best model is selected from the measured test results.

Because this is medical imaging, the resulting system should be treated as an AI-assisted research prototype. Before any clinical use, it would require external validation, representative patient cohorts, robust bias and calibration testing, explainability review, privacy/security controls, regulatory compliance and evaluation by qualified medical professionals.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***